# KV Cache — Interactive Demo (Refined)

This notebook walks you through the key ideas behind **KV Cache** in Transformer-based language models:

1. Why caching keys and values speeds up autoregressive generation
2. How much memory KV Cache consumes
3. Variants: **Multi-query Attention**, **Group-query Attention**, **Multi-head Latent Attention**
4. **Sliding Window Attention** and **Streaming LLM**
5. **Pruning** the KV Cache (Scissorhands / H2O)
6. **Cross-conversation** cache reuse

Run every cell from top to bottom; interactive widgets appear where you see sliders or buttons.

## Learning Objectives

After finishing this notebook you should be able to:

- Explain why KV Cache reduces *redundant* K/V computation during decoding (and what it does **not** save).
- Estimate the memory footprint of a KV Cache for a given model and sequence length.
- Compare MHA, MQA, GQA, and MLA and articulate their trade-offs.
- Describe how sliding-window and pruning-based methods shrink the cache at runtime.

## 0 · Setup

In [ ]:
import math
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

plt.rcParams.update({'figure.dpi': 110, 'font.size': 11})
print('Setup complete \u2713')

---
## 1 · Scaled Dot-Product Attention (recap)

Given token embeddings $x_1, \ldots, x_T$ we compute:

$$
Q = XW_Q, \quad K = XW_K, \quad V = XW_V
$$
$$
\text{Attention}(Q, K, V) = \text{softmax}\!\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

During **autoregressive generation** the model produces one token at a time.
At step $t$ the new query is $q_t$, but the keys/values for tokens $1 \ldots t-1$ are **identical** to what was computed in previous steps — without KV Cache we recompute them from scratch every step.

> **Predict first:** Before running the next cell, sketch what a 4-token causal attention weight matrix should look like. Where are the zeros?

In [ ]:
def softmax(x):
    e = np.exp(x - x.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)

def scaled_dot_product_attention(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.T / math.sqrt(d_k)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores)
    return weights @ V, weights

rng = np.random.default_rng(42)
T, d = 4, 8
X  = rng.standard_normal((T, d))
Wq = rng.standard_normal((d, d)) * 0.3
Wk = rng.standard_normal((d, d)) * 0.3
Wv = rng.standard_normal((d, d)) * 0.3

Q, K, V = X @ Wq, X @ Wk, X @ Wv
causal_mask = np.tril(np.ones((T, T), dtype=bool))
output, weights = scaled_dot_product_attention(Q, K, V, mask=causal_mask)

fig, ax = plt.subplots(figsize=(4, 3.5))
im = ax.imshow(weights, cmap='Blues', vmin=0, vmax=1)
ax.set_title('Causal attention weights\n(4-token sequence)')
ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
ax.set_xticks(range(T)); ax.set_yticks(range(T))
fig.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout()
plt.show()

---
### ⚠️ Common Misconception — What KV Cache Does *Not* Save

**KV Cache eliminates the re-projection** of past tokens' keys and values ($W_K x_i$, $W_V x_i$) for tokens $i < t$.

It does **not** eliminate the attention score computation: at every decode step, query $q_t$ must attend to **all $t$ cached keys**, so attention still costs $O(t)$ dot-products per step. Total decode cost remains $O(T^2)$ in attention operations, but drops from $O(T^2)$ to $O(T)$ in K/V projection cost.

This is why long-context models still benefit from optimisations like Sliding Window Attention even when KV Cache is active.

---
## 2 · KV Cache — Eliminating Redundant K/V Projections

**Without cache:** generating token $t$ requires computing $K$ and $V$ for all $t$ positions → $O(t)$ matrix multiplications per step → $O(T^2)$ total K/V work.

**With cache:** $K$ and $V$ up to position $t-1$ are stored from the previous step.
Only the *new* key/value pair is computed and appended → one matmul per step → $O(T)$ total K/V work.

> **Mini-exercise:** Move the slider to step 8 and note the "total computations" counter. Verify it by hand: without cache the total is $1+2+\ldots+8 = ?$. With cache it should be exactly $8$.

In [ ]:
def draw_kvcache_comparison(step, total_steps=8):
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    colors_cached  = ['#4CAF50'] * step + ['#E0E0E0'] * (total_steps - step)
    colors_nocache = ['#F44336'] * step + ['#E0E0E0'] * (total_steps - step)

    for ax, title, cols, cost_label in [
        (axes[0], 'WITHOUT KV Cache', colors_nocache,
         f'K/V projections at step {step}: {step}'),
        (axes[1], 'WITH KV Cache',    colors_cached,
         f'K/V projections at step {step}: 1 (new token only)'),
    ]:
        for i in range(total_steps):
            ax.barh(i, 1, color=cols[i], edgecolor='white', linewidth=1.5)
        ax.set_title(title, fontweight='bold')
        ax.set_xlim(0, 1.15)
        ax.set_yticks(range(total_steps))
        ax.set_yticklabels([f'tok {i+1}' for i in range(total_steps)])
        ax.set_xlabel('In cache / computed this step')
        ax.text(0.5, -1.6, cost_label, ha='center', color='#333', fontsize=10,
                transform=ax.get_yaxis_transform())

    total_no_cache = step * (step + 1) // 2
    expected = total_steps * (total_steps + 1) // 2
    fig.suptitle(
        f'Decode step {step}/{total_steps}  |  '
        f'Cumulative K/V projections — no cache: {total_no_cache}   cache: {step}',
        fontsize=12, fontweight='bold', y=1.02)
    plt.tight_layout()
    plt.show()

step_slider = widgets.IntSlider(value=1, min=1, max=8, step=1,
                                description='Decode step:',
                                continuous_update=False,
                                style={'description_width': 'initial'})
widgets.interact(draw_kvcache_comparison, step=step_slider, total_steps=widgets.fixed(8))

---
## 3 · KV Cache Memory Footprint

For a transformer with $L$ layers, $H$ heads, $d_\text{head}$ head dimension,
$P$ bytes per value (e.g. 2 for FP16), and $T$ cached tokens:

$$
\text{KV Cache} = \underbrace{2}_{\text{K + V}} \times L \times H \times d_\text{head} \times P \times T \;\text{bytes}
$$

> **Note:** all sizes below use **binary** units (1 KiB = 1024 B, 1 GiB = 1024³ B),
> matching how GPU VRAM is typically reported.

**Example — Gemma 2:** $L{=}46$, $H{=}32$, $d_\text{head}{=}128$, FP16 → **736 KiB per token**.

> **Predict first:** For Gemma 2 on an 80 GiB A100, roughly how many tokens can fit in the KV cache if the whole GPU were available? Slide to check.

In [ ]:
out_memory = widgets.Output()

w_layers = widgets.IntSlider(value=46,  min=1,   max=128,   description='Layers (L):')
w_heads  = widgets.IntSlider(value=32,  min=1,   max=128,   description='Heads (H):')
w_dhead  = widgets.IntSlider(value=128, min=16,  max=512, step=16, description='d_head:')
w_prec   = widgets.Dropdown(options=[('FP32 (4 B)', 4), ('FP16 (2 B)', 2), ('INT8 (1 B)', 1)],
                             value=2, description='Precision:')
w_tokens = widgets.IntSlider(value=4096, min=128, max=131072, step=128,
                              description='Seq len (T):',
                              style={'description_width': 'initial'})
w_gpu_gb = widgets.FloatSlider(value=80, min=8, max=640, step=8,
                                description='GPU (GiB):')

def update_memory(layers, heads, dhead, precision_bytes, tokens, gpu_gb):
    per_token_bytes = 2 * layers * heads * dhead * precision_bytes
    total_bytes     = per_token_bytes * tokens
    gpu_bytes       = gpu_gb * (1024 ** 3)
    per_token_kib   = per_token_bytes / 1024
    total_gib       = total_bytes / (1024 ** 3)
    max_tokens      = int(gpu_bytes / per_token_bytes)
    frac            = min(total_bytes / gpu_bytes, 1.0)

    with out_memory:
        clear_output(wait=True)
        fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))

        # Left: KV cache total vs GPU VRAM — both in GiB (consistent units)
        bar_vals   = [total_gib, gpu_gb]
        bar_labels = [f'KV Cache total\n({tokens:,} tokens)', f'GPU VRAM\n({gpu_gb:.0f} GiB)']
        bar_colors = ['#1976D2', '#FFA000']
        bars = axes[0].bar(bar_labels, bar_vals, color=bar_colors,
                           edgecolor='white', width=0.45)
        axes[0].set_ylabel('Size (GiB)  [1 GiB = 1024\u00b3 bytes]')
        axes[0].set_title('KV Cache vs GPU capacity')
        for b, v in zip(bars, bar_vals):
            axes[0].text(b.get_x() + b.get_width() / 2, v * 1.02 + 1e-6,
                         f'{v:.2f} GiB', ha='center', fontsize=9)
        axes[0].set_ylim(0, max(bar_vals) * 1.2 + 1e-6)

        # Right: donut — fraction of GPU used
        axes[1].pie([frac, 1 - frac],
                    labels=['KV Cache', 'Remaining GPU'],
                    colors=['#1976D2', '#E0E0E0'],
                    startangle=90, autopct='%1.1f%%',
                    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5})
        axes[1].set_title(f'KV cache fraction of GPU\n(per-token: {per_token_kib:.1f} KiB)')

        fig.suptitle(
            f'Per-token: {per_token_kib:.1f} KiB  |  '
            f'Total: {total_gib:.3f} GiB  |  '
            f'Max tokens on this GPU: {max_tokens:,}',
            fontsize=11, fontweight='bold')
        plt.tight_layout()
        plt.show()

ui = widgets.VBox([
    widgets.HBox([w_layers, w_heads]),
    widgets.HBox([w_dhead,  w_prec]),
    widgets.HBox([w_tokens, w_gpu_gb]),
    out_memory,
])
widgets.interactive_output(update_memory, {
    'layers': w_layers, 'heads': w_heads, 'dhead': w_dhead,
    'precision_bytes': w_prec, 'tokens': w_tokens, 'gpu_gb': w_gpu_gb,
})
display(ui)

---
## 4 · Attention Variants: MHA → MQA → GQA → MLA

| Method | Keys/Values | KV Cache size | Model quality |
|---|---|---|---|
| **MHA** – Multi-Head Attention | one K,V per head | full | best |
| **MQA** – Multi-Query Attention | shared single K,V | ÷ H | may degrade |
| **GQA** – Group-Query Attention | one K,V per group | ÷ n_groups | good trade-off |
| **MLA** – Multi-head Latent Attention | compressed latent | very small | DeepSeek |

> **Remainder heads (GQA):** When `n_heads` is not evenly divisible by `group_size`,
> ceiling division gives `⌈n_heads / group_size⌉` groups. The last group receives the
> leftover heads (fewer than `group_size`). For example, 10 heads with group_size=8 →
> 2 groups: group 0 gets heads 0–7, group 1 gets heads 8–9. Both groups still share
> exactly one K/V pair each.

> **Predict first:** With 8 heads and group_size=4, how many K/V pairs does GQA need?
> Try it in the widget, then try the non-divisible case: 10 heads, group_size=8.

In [ ]:
def draw_variants(n_heads=8, group_size=2, mla_rank=4, d_head=64, seq_len=512, prec_bytes=2):
    # ── Ceiling-based grouping ─────────────────────────────────────────────
    n_groups         = math.ceil(n_heads / group_size)
    group_membership = [min(h // group_size, n_groups - 1) for h in range(n_heads)]
    group_members    = {g: [h for h in range(n_heads) if group_membership[h] == g]
                        for g in range(n_groups)}

    mha_kv = 2 * n_heads  * d_head * seq_len * prec_bytes / 1024
    mqa_kv = 2 * 1        * d_head * seq_len * prec_bytes / 1024
    gqa_kv = 2 * n_groups * d_head * seq_len * prec_bytes / 1024
    mla_kv = 2 * mla_rank * seq_len * prec_bytes / 1024

    labels = ['MHA', 'GQA', 'MQA', 'MLA']
    sizes  = [mha_kv, gqa_kv, mqa_kv, mla_kv]
    colors = ['#1565C0', '#2E7D32', '#F57F17', '#6A1B9A']

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))

    # Bar chart (left)
    bars = axes[0].bar(labels, sizes, color=colors, edgecolor='white', width=0.55)
    axes[0].set_ylabel('KV Cache (KiB) per layer')
    axes[0].set_title(f'KV Cache — {n_heads} heads, seq={seq_len}, d_head={d_head}')
    for b, s in zip(bars, sizes):
        axes[0].text(b.get_x() + b.get_width() / 2, b.get_height() * 1.02 + 1e-6,
                     f'{s:.1f} KiB', ha='center', fontsize=9)
    axes[0].set_ylim(0, max(sizes) * 1.22)

    # Head-to-group diagram (right)
    ax = axes[1]
    ax.set_xlim(-0.6, 3.6)
    ax.set_ylim(-0.7, n_heads + 0.3)
    ax.set_title('Query heads → KV group mapping (GQA)')
    ax.axvline(0.5, color='gray', lw=0.5, ls='--')
    ax.axvline(1.5, color='gray', lw=0.5, ls='--')
    ax.text(0,   n_heads + 0.1, 'Q heads', ha='center', fontsize=9, color='#333')
    ax.text(1,   n_heads + 0.1, 'K groups', ha='center', fontsize=9, color='#333')
    ax.text(2,   n_heads + 0.1, 'V groups', ha='center', fontsize=9, color='#333')

    palette = plt.cm.tab10(np.linspace(0, 0.9, max(n_groups, 1)))

    # Q head boxes
    for h in range(n_heads):
        g   = group_membership[h]
        col = palette[g % len(palette)]
        ax.add_patch(mpatches.FancyBboxPatch(
            (-.38, h - .38), 0.76, 0.76, color=col, alpha=0.85,
            boxstyle='round,pad=0.04'))
        ax.text(0, h, f'Q{h}', ha='center', va='center',
                fontsize=8, color='white', fontweight='bold')

    # KV group boxes (y_center from actual member positions → correct for remainders)
    for g in range(n_groups):
        members  = group_members[g]
        y_center = (members[0] + members[-1]) / 2.0
        col      = palette[g % len(palette)]
        for x_pos, label in [(1, f'K{g}'), (2, f'V{g}')]:
            ax.add_patch(mpatches.FancyBboxPatch(
                (x_pos - .38, y_center - .38), 0.76, 0.76,
                color=col, alpha=0.7, boxstyle='round,pad=0.04'))
            ax.text(x_pos, y_center, label, ha='center', va='center',
                    fontsize=8, color='white', fontweight='bold')
        for h in members:
            ax.plot([0.38, 0.62], [h, y_center], color=col, lw=1, alpha=0.55)

    ax.axis('off')
    plt.tight_layout()
    plt.show()

    print(f'  n_groups (ceil)={n_groups} | Reduction vs MHA → '
          f'GQA: x{mha_kv/gqa_kv:.1f}  '
          f'MQA: x{mha_kv/max(mqa_kv, 1e-9):.1f}  '
          f'MLA: x{mha_kv/max(mla_kv, 1e-9):.1f}')

widgets.interact(
    draw_variants,
    n_heads    = widgets.IntSlider(value=8,   min=1,  max=32,  step=1,
                                   description='# Heads:'),
    group_size = widgets.IntSlider(value=2,   min=1,  max=16,  step=1,
                                   description='Group size:'),
    mla_rank   = widgets.IntSlider(value=4,   min=1,  max=32,  step=1,
                                   description='MLA rank:'),
    d_head     = widgets.IntSlider(value=64,  min=16, max=256, step=16,
                                   description='d_head:'),
    seq_len    = widgets.IntSlider(value=512, min=64, max=8192, step=64,
                                   description='Seq len:'),
    prec_bytes = widgets.Dropdown(options=[('FP32',4),('FP16',2),('INT8',1)],
                                  value=2, description='Precision:'),
)

---
## 5 · Sliding Window Attention

Instead of every query attending to **all** past tokens, each query attends only to the **W nearest** tokens.
This caps the KV cache size at $W$ entries per layer regardless of total sequence length.

Used by **Mistral 7B** and **GPT-OSS** variants.

In [ ]:
def draw_sliding_window(seq_len=16, window=5, current_query=10):
    current_query = min(current_query, seq_len - 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, title, use_window in [
        (axes[0], 'Full (causal) attention', False),
        (axes[1], f'Sliding window  W={window}', True),
    ]:
        mat = np.zeros((seq_len, seq_len))
        for q in range(seq_len):
            lo = max(0, q - window + 1) if use_window else 0
            mat[q, lo:q+1] = 0.5
        q  = current_query
        lo = max(0, q - window + 1) if use_window else 0
        mat[q, lo:q+1] = 1.0
        ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
        ax.set_title(title, fontweight='bold')
        ax.set_ylabel('Query position')
        ax.axhline(current_query - 0.5, color='#1565C0', lw=1.5, ls='--')
        ax.axhline(current_query + 0.5, color='#1565C0', lw=1.5, ls='--')
        n_attended = (current_query + 1) if not use_window else min(window, current_query + 1)
        ax.set_xlabel(f'Keys attended at query {current_query}: {n_attended}')
    plt.suptitle(f'Seq len={seq_len}, highlighted query={current_query}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(
    draw_sliding_window,
    seq_len       = widgets.IntSlider(value=16, min=8,  max=32, step=1,
                                      description='Seq len:'),
    window        = widgets.IntSlider(value=5,  min=2,  max=16, step=1,
                                      description='Window W:'),
    current_query = widgets.IntSlider(value=10, min=0,  max=31, step=1,
                                      description='Query idx:'),
)

### 5b · Streaming LLM

**Streaming LLM** keeps a small set of *sink* tokens at the very beginning (the model heavily attends
to these even when they appear irrelevant) plus a sliding window of recent tokens.
This avoids the perplexity collapse that occurs when the window simply drops early tokens.

In [ ]:
def draw_streaming_llm(seq_len=20, n_sink=4, window=6, current_query=14):
    current_query = min(current_query, seq_len - 1)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    for ax, title, use_sink in [
        (axes[0], f'Sliding window only  W={window}', False),
        (axes[1], f'Streaming LLM  sink={n_sink}  W={window}', True),
    ]:
        mat = np.zeros((seq_len, seq_len))
        for q in range(seq_len):
            lo = max(0, q - window + 1)
            mat[q, lo:q+1] = 0.5
            if use_sink:
                mat[q, :min(n_sink, q+1)] = 0.75
        q  = current_query
        lo = max(0, q - window + 1)
        mat[q, lo:q+1] = 1.0
        if use_sink:
            mat[q, :min(n_sink, q+1)] = 0.9
        ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=1, aspect='auto')
        ax.set_title(title, fontweight='bold')
        ax.set_xlabel('Key position'); ax.set_ylabel('Query position')
        ax.axhline(current_query - 0.5, color='#1565C0', lw=1.5, ls='--')
        ax.axhline(current_query + 0.5, color='#1565C0', lw=1.5, ls='--')
        if use_sink and n_sink > 0:
            ax.axvline(n_sink - 0.5, color='green', lw=1.5, ls=':')
            ax.text(n_sink / 2 - 0.5, -0.9, 'sinks',
                    ha='center', color='green', fontsize=9)
    plt.suptitle(f'Seq len={seq_len}, query={current_query}',
                 fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(
    draw_streaming_llm,
    seq_len       = widgets.IntSlider(value=20, min=10, max=40, step=1,
                                      description='Seq len:'),
    n_sink        = widgets.IntSlider(value=4,  min=0,  max=10, step=1,
                                      description='Sink tokens:'),
    window        = widgets.IntSlider(value=6,  min=2,  max=15, step=1,
                                      description='Window W:'),
    current_query = widgets.IntSlider(value=14, min=0,  max=39, step=1,
                                      description='Query idx:'),
)

---
## 6 · Pruning the KV Cache (Scissorhands / H2O)

Key observation: **most tokens receive very little attention**; only a small fraction
("heavy hitters") accumulate large attention scores repeatedly.

**Scissorhands** and **H2O** evict low-importance tokens from the KV cache and retain only
the heavy hitters plus recent tokens.

> **Predict first:** If you keep only the top 10% of tokens plus the last 3 recent tokens in
> a 20-token cache, how many K/V entries remain? Would quality suffer more for short or long sequences?

In [ ]:
def simulate_pruning(seq_len=20, keep_ratio=0.4, recent_tokens=3, seed=7):
    # ── Safety clamps (handles extreme slider combos) ──────────────────────
    recent_tokens = int(np.clip(recent_tokens, 0, seq_len - 1))
    n_non_recent  = seq_len - recent_tokens          # always >= 1
    keep_ratio    = float(np.clip(keep_ratio, 0.0, 1.0))

    rng_local = np.random.default_rng(int(seed))
    # Heavy-tailed scores for non-recent tokens
    raw       = rng_local.exponential(scale=1.0, size=n_non_recent)
    score_sum = raw.sum()
    norm_scores = raw / score_sum if score_sum > 0 else np.zeros(n_non_recent)

    n_keep = int(np.clip(round(n_non_recent * keep_ratio), 0, n_non_recent))

    ranked        = np.argsort(norm_scores)[::-1]
    heavy_hitters = set(ranked[:n_keep].tolist())
    recents       = set(range(n_non_recent, seq_len))    # last recent_tokens positions
    kept          = heavy_hitters | recents
    evicted       = set(range(seq_len)) - kept

    # Build display score array (recents shown as 0 — they don't compete in ranking)
    display_scores = np.concatenate([norm_scores, np.zeros(recent_tokens)])

    fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))

    # Left: scores
    bar_colors = [
        '#2196F3' if i in recents else
        '#4CAF50' if i in heavy_hitters else
        '#EF5350'
        for i in range(seq_len)
    ]
    axes[0].bar(range(seq_len), display_scores, color=bar_colors, edgecolor='white')
    axes[0].set_title(
        'Accumulated attention scores per token\n'
        '(last N tokens always kept \u2014 shown with score=0 by convention)')
    axes[0].set_xlabel('Token position')
    axes[0].set_ylabel('Score (normalised, non-recent tokens)')
    patches = [
        mpatches.Patch(color='#4CAF50', label=f'Heavy hitter (top {keep_ratio:.0%}, kept)'),
        mpatches.Patch(color='#2196F3', label=f'Recent (last {recent_tokens} tok, always kept)'),
        mpatches.Patch(color='#EF5350', label='Evicted'),
    ]
    axes[0].legend(handles=patches, fontsize=8)

    # Right: cache state
    state_colors = [
        '#4CAF50' if i in heavy_hitters else
        '#2196F3' if i in recents else
        '#EF5350'
        for i in range(seq_len)
    ]
    axes[1].bar(range(seq_len), [1] * seq_len, color=state_colors, edgecolor='white')
    axes[1].set_title(
        f'KV Cache after pruning — kept {len(kept)}/{seq_len} '
        f'({100 * len(kept) / seq_len:.0f}%)')
    axes[1].set_xlabel('Token position')
    axes[1].set_yticks([])

    plt.tight_layout()
    plt.show()

widgets.interact(
    simulate_pruning,
    seq_len       = widgets.IntSlider(value=20,  min=5,   max=60,  step=1,
                                      description='Seq len:'),
    keep_ratio    = widgets.FloatSlider(value=0.4, min=0.0, max=1.0, step=0.05,
                                        description='Keep ratio:'),
    recent_tokens = widgets.IntSlider(value=3,   min=0,   max=20,  step=1,
                                      description='Recent kept:'),
    seed          = widgets.IntSlider(value=7,   min=0,   max=99,  step=1,
                                      description='Random seed:'),
)

---
## 7 · Cross-Conversation Cache Reuse

When the **system prompt** is identical across requests, a server can cache its KV representations
and reuse them — computing only the variable user suffix each time.

**Key design rule:** put stable content first, variable content last — cache hits require a common prefix.

```
[System prompt — stable]  [Memory/tools — semi-stable]  [User message — variable]
  ←──────────── CACHE HIT ────────────→  ← compute only this →
```

In [ ]:
def draw_prompt_cache(system_tokens=2000, user_tokens=200, n_requests=50, cost_per_1k=0.003):
    total_no_cache   = n_requests * (system_tokens + user_tokens)
    total_with_cache = system_tokens + n_requests * user_tokens
    saved_tokens     = total_no_cache - total_with_cache
    saved_dollars    = saved_tokens / 1000 * cost_per_1k

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    cats      = ['No cache', 'Prefix cache']
    sys_vals  = [n_requests * system_tokens, system_tokens]
    user_vals = [n_requests * user_tokens,   n_requests * user_tokens]
    x = np.arange(2)
    axes[0].bar(x, sys_vals,  label='System tokens (computed)', color='#F44336', edgecolor='white')
    axes[0].bar(x, user_vals, bottom=sys_vals, label='User tokens', color='#42A5F5', edgecolor='white')
    axes[0].set_xticks(x); axes[0].set_xticklabels(cats)
    axes[0].set_ylabel('Total tokens processed')
    axes[0].set_title(f'{n_requests} requests')
    axes[0].legend(fontsize=9)
    for xi, (s, u) in enumerate(zip(sys_vals, user_vals)):
        axes[0].text(xi, (s+u)*1.01, f'{(s+u):,}', ha='center', fontsize=9)

    axes[1].axis('off')
    summary = (
        f"Without cache     : {total_no_cache:>12,} tokens\n"
        f"With prefix cache : {total_with_cache:>12,} tokens\n"
        f"\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n"
        f"Tokens saved      : {saved_tokens:>12,}\n"
        f"Cost saved        : ${saved_dollars:>11.2f}\n"
        f"  (@ ${cost_per_1k}/1k tokens)\n\n"
        f"Cache efficiency  : {100*(1 - total_with_cache/total_no_cache):.1f}% tokens skipped"
    )
    axes[1].text(0.05, 0.5, summary, transform=axes[1].transAxes,
                 fontsize=12, va='center', fontfamily='monospace',
                 bbox=dict(boxstyle='round', facecolor='#E3F2FD', alpha=0.8))
    plt.suptitle('Prompt prefix caching — cost savings', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

widgets.interact(
    draw_prompt_cache,
    system_tokens = widgets.IntSlider(value=2000, min=100,  max=50000, step=100,
                                       description='System tokens:',
                                       style={'description_width': 'initial'}),
    user_tokens   = widgets.IntSlider(value=200,  min=10,   max=5000,  step=10,
                                       description='User tokens:'),
    n_requests    = widgets.IntSlider(value=50,   min=2,    max=1000,  step=1,
                                       description='# Requests:'),
    cost_per_1k   = widgets.FloatSlider(value=0.003, min=0.0001, max=0.05, step=0.0005,
                                         readout_format='.4f', description='$/1k tokens:'),
)

---
## 8 · Summary Table

In [ ]:
html = """
<style>
  table{border-collapse:collapse;width:100%;font-size:13px}
  th{background:#1565C0;color:white;padding:8px 10px;text-align:left}
  tr:nth-child(even){background:#E3F2FD}
  td{padding:7px 10px;border-bottom:1px solid #ccc}
  .y{color:#2E7D32;font-weight:bold} .n{color:#B71C1C;font-weight:bold}
</style>
<table>
<tr><th>Method</th><th>Core idea</th><th>Changes Attention?</th><th>Needs fine-tuning?</th><th>Trade-off</th></tr>
<tr><td>Flash Attention</td><td>Move less data between HBM&#8596;SRAM</td><td class='n'>No</td><td class='n'>No</td><td>Tiny extra compute</td></tr>
<tr><td><b>KV Cache</b></td><td>Store K,V; skip re-projection at each step</td><td class='n'>No</td><td class='n'>No</td><td>Memory grows with seq len</td></tr>
<tr><td>Multi-Query Attention</td><td>All queries share one K,V</td><td class='y'>Yes</td><td class='y'>Yes</td><td>May hurt model quality</td></tr>
<tr><td>Group-Query Attention</td><td>Groups share K,V (Llama, Gemma)</td><td class='y'>Yes</td><td class='y'>Yes</td><td>Good quality&#8211;memory trade-off</td></tr>
<tr><td>Multi-head Latent Attention</td><td>Compress K,V to low-rank latent (DeepSeek)</td><td class='y'>Yes</td><td class='y'>Yes</td><td>Tiny cache, strong results</td></tr>
<tr><td>Sliding Window Attention</td><td>Attend only to W recent tokens</td><td class='y'>Yes</td><td>?</td><td>Loses long-range context</td></tr>
<tr><td>Streaming LLM</td><td>Sliding window + sink tokens</td><td class='y'>Yes</td><td>?</td><td>Better than pure window</td></tr>
<tr><td>Pruning KV Cache</td><td>Evict low-attention tokens at runtime</td><td class='y'>Yes</td><td class='n'>No</td><td>Risk on rare-pattern queries</td></tr>
<tr><td>Speculative Decoding</td><td>Small model drafts; large model verifies</td><td class='n'>No (in theory)</td><td class='n'>No</td><td>Extra compute for draft model</td></tr>
</table>
"""
display(HTML(html))

---
## 9 · Student Reflection Questions

1. **Memory vs. compute trade-off:** KV Cache saves K/V projection work but costs memory.
   At what sequence length does the KV cache exceed half the GPU VRAM for Gemma 2 on an 80 GiB A100?
   (Use the memory widget to find the answer.)

2. **GQA group size:** With 32 heads and group_size=4 → 8 groups. If you change to group_size=5,
   how many groups does ceiling division give? Check with the widget.

3. **Sliding window vs. full attention:** Name a task where sliding window attention would clearly fail.
   Name one where it would be fine. Explain why in each case.

4. **Sink tokens:** Why do language models attend heavily to the very first tokens even when they are
   irrelevant (e.g., the `<bos>` token)? (Hint: think about what happens to the softmax when no
   good key exists in the window.)

5. **Cross-conversation caching:** You're building an AI agent with a 10,000-token system prompt and
   50-token user messages. How should you structure the prompt to maximise cache hits?

6. **Mini-exercise — pruning with keep_ratio=0.0:**
   Set keep_ratio to 0 in the pruning widget (only recent tokens kept). Observe: which tokens remain?
   Now set recent_tokens=1. What is the minimum possible cache size? Does the first bar chart change?
   Expected: bar chart unchanged (scores are independent of keep_ratio); only the state chart changes.

---
## 10 · Suggested Extensions

- **Implement a tiny GPT with KV Cache** using PyTorch and measure wall-clock speedup vs. the naive
  version for sequences 64 → 1024 tokens.
- **Reproduce H2O Figure 1:** plot perplexity vs. cache budget fraction for a small LM on a text dataset.
- **Benchmark MHA vs. GQA:** fine-tune a small transformer with different group sizes and plot
  accuracy vs. inference memory.
- **Implement Streaming LLM:** feed a model tokens one-by-one and maintain the sink + sliding-window
  cache manually; measure perplexity degradation vs. standard KV cache.
- **Prompt-caching API:** use the Anthropic or OpenAI API to measure actual latency reduction from
  prefix caching on a repeated system prompt.

---
## 11 · Sanity-Check / Validation (Instructor Cell)

Run this cell to verify correctness of the key fixed functions across edge cases.

In [ ]:
# ── Validation: GQA ceiling grouping ─────────────────────────────────────
import math, numpy as np

def _n_groups(n_heads, group_size):
    return math.ceil(n_heads / group_size)

cases_gqa = [
    (8,  2, 4),   # evenly divisible
    (8,  3, 3),   # 8/3 = 2.67 → 3 groups
    (10, 8, 2),   # remainder: 10/8 → 2 groups (8+2)
    (4,  4, 1),   # single group
    (1,  3, 1),   # single head
]
print("=== GQA group count (ceil) ===")
for n_heads, gs, expected in cases_gqa:
    got = _n_groups(n_heads, gs)
    status = "OK" if got == expected else f"FAIL (expected {expected})"
    print(f"  n_heads={n_heads}, group_size={gs} → {got} groups  {status}")

# ── Validation: pruning edge cases ────────────────────────────────────────
def _prune_stats(seq_len, keep_ratio, recent_tokens, seed=7):
    recent_tokens = int(np.clip(recent_tokens, 0, seq_len - 1))
    n_non_recent  = seq_len - recent_tokens
    keep_ratio    = float(np.clip(keep_ratio, 0.0, 1.0))
    rng           = np.random.default_rng(seed)
    raw           = rng.exponential(scale=1.0, size=n_non_recent)
    score_sum     = raw.sum()
    norm_scores   = raw / score_sum if score_sum > 0 else np.zeros(n_non_recent)
    n_keep        = int(np.clip(round(n_non_recent * keep_ratio), 0, n_non_recent))
    heavy         = set(np.argsort(norm_scores)[::-1][:n_keep].tolist())
    recents       = set(range(n_non_recent, seq_len))
    kept          = heavy | recents
    return len(kept), seq_len, not np.any(np.isnan(norm_scores))

print("\n=== Pruning edge cases ===")
prune_cases = [
    (20, 0.4,  3),
    (20, 0.0,  3),   # keep_ratio=0
    (20, 1.0,  3),   # keep all non-recent
    (20, 0.4, 19),   # recent near seq_len
    (5,  0.5,  4),   # tiny seq, large recent
    (10, 0.5,  9),   # recent = seq_len-1
]
all_ok = True
for sl, kr, rt in prune_cases:
    kept, total, no_nan = _prune_stats(sl, kr, rt)
    ok = kept <= total and no_nan
    all_ok = all_ok and ok
    print(f"  seq={sl}, keep={kr}, recent={rt} → kept={kept}/{total}  {'OK' if ok else 'FAIL'}")

print(f"\nAll validation checks passed: {all_ok}")